<a href="https://colab.research.google.com/github/omarelmalak/csc490-group2/blob/main/Logo_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLOv8 Logo Detection End-to-End Pipeline

### Step 1: Environment Setup and Library Imports

In [1]:
# Install Ultralytics library for YOLOv8 training
!pip install -q ultralytics

import os
import shutil
import glob
import yaml
import random
import xml.etree.ElementTree as ET
from collections import defaultdict
from pathlib import Path
from ultralytics import YOLO

print("Environment ready and libraries imported!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 9.3 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Environment ready and libraries imported!


### Step 2: Mount Google Drive
Run this cell to link your Google Drive so the script can access your zipped dataset.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Step 3: Extract Logo Dataset
Extracts your zipped dataset directly from Google Drive into the local session storage.

In [3]:
# Unzip the logo dataset from Google Drive into /content/logos
!unzip -q "/content/drive/MyDrive/logodet.zip" -d "/content/logos"

# Inspect the extracted directory structure to verify it matches what we need
print("Extracted contents of /content/logos:")
for root, dirs, files in os.walk("/content/logos"):
    level = root.replace("/content/logos", '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files[:3]:  # Print up to 3 files as preview
        print(f"{subindent}{f}")
    if len(files) > 3:
        print(f"{subindent}... and {len(files) - 3} more files")

Streaming output truncated to the last 5000 lines.
                ... and 93 more files
            Freia/
                58.xml
                77.xml
                10.jpg
                ... and 137 more files
            laura secord chocolates/
                43.xml
                58.xml
                77.xml
                ... and 123 more files
            shengfeng/
                38.xml
                38.jpg
                60.xml
                ... and 13 more files
            coffeeheaven/
                10.jpg
                69.jpg
                60.xml
                ... and 35 more files
            Cows/
                59.jpg
                62.xml
                13.xml
                ... and 7 more files
            chenkeming-1/
                33.jpg
                38.xml
                30.jpg
                ... and 29 more files
            maarud/
                43.xml
                58.xml
                10.jpg
                ... and 107 mo

### Step 4: Convert Pascal XML Annotations to YOLO TXT Format
Parses the XML annotations in the "Other" folder and converts them to standard normalized YOLO `.txt` coordinate files next to each image.

In [4]:
# --- CONFIGURATION ---
# Convert only the XML files found in the 'Others' category folder under LogoDet-3K
SOURCE_XML_DIR = "/content/logos/LogoDet-3K/Others"

# Automatically detect classes from XML files to build a mapping
classes = []
xml_files = glob.glob(os.path.join(SOURCE_XML_DIR, "**/*.xml"), recursive=True)

for xml_file in xml_files:
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        for obj in root.iter('object'):
            cls_name = obj.find('name').text
            if cls_name not in classes:
                classes.append(cls_name)
    except Exception:
        continue

classes = sorted(classes)
class_to_id = {name: i for i, name in enumerate(classes)}
print(f"Detected classes inside 'Others': {len(classes)} classes found.")
if len(classes) > 0:
    print(f"Sample classes: {classes[:10]}")

def convert_coords(size, box):
    # Convert absolute pixel coordinates to normalized center coordinates
    dw = 1.0 / size[0] if size[0] > 0 else 1.0
    dh = 1.0 / size[1] if size[1] > 0 else 1.0
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

# Convert each XML to TXT in its respective directory
converted_count = 0
for xml_file in xml_files:
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        # Retrieve image dimensions
        size = root.find('size')
        w = int(size.find('width').text)
        h = int(size.find('height').text)

        if w == 0 or h == 0:
            continue # Skip invalid size

        txt_filename = os.path.splitext(os.path.basename(xml_file))[0] + ".txt"
        # Save TXT alongside the XML directly in the same subfolder
        txt_output_path = os.path.join(os.path.dirname(xml_file), txt_filename)

        with open(txt_output_path, 'w') as out_file:
            for obj in root.iter('object'):
                cls_name = obj.find('name').text
                if cls_name not in class_to_id:
                    continue
                cls_id = class_to_id[cls_name]
                xmlbox = obj.find('bndbox')
                b = (
                    float(xmlbox.find('xmin').text),
                    float(xmlbox.find('xmax').text),
                    float(xmlbox.find('ymin').text),
                    float(xmlbox.find('ymax').text)
                )
                bb = convert_coords((w, h), b)
                out_file.write(f"{cls_id} {' '.join([f'{coord:.6f}' for coord in bb])}\n")
        converted_count += 1
    except Exception as e:
        print(f"Failed to process {xml_file}: {e}")

print(f"Successfully converted {converted_count} XML files to YOLO TXT format inside 'Others'!")

Detected classes inside 'Others': 373 classes found.
Sample classes: ['7Days Inn', 'A T CROSS', 'ACE Team', 'ACN', 'AIST', 'AKPET', 'AMAT', 'ANLI', 'AP Industries', 'APCO']
Successfully converted 15513 XML files to YOLO TXT format inside 'Others'!


### Step 5: Define Paths & Split Dataset (80% Train, 10% Val, 10% Test)
This section processes only the "Other" folder, matches the images and converted `.txt` label files, and distributes them proportionally.

In [5]:
# --- PATH CONFIGURATION ---
# Target only the 'Others' category folder under LogoDet-3K
SOURCE_IMAGES_DIR = "/content/logos/LogoDet-3K/Others"
SOURCE_LABELS_DIR = "/content/logos/LogoDet-3K/Others"

# Destination directories for YOLO format
DATASET_ROOT = "/content/logo_yolo_dataset"

# Create train/val/test directories for images and labels
for split_type in ['train', 'val', 'test']:
    os.makedirs(os.path.join(DATASET_ROOT, 'images', split_type), exist_ok=True)
    os.makedirs(os.path.join(DATASET_ROOT, 'labels', split_type), exist_ok=True)

# Find all images inside the 'Others' category subfolders
image_extensions = ('*.png', '*.jpg', '*.jpeg', '*.JPG', '*.JPEG', '*.PNG')
all_images = []
for ext in image_extensions:
    all_images.extend(glob.glob(os.path.join(SOURCE_IMAGES_DIR, "**", ext), recursive=True))

# Group matching image-label pairs by their logo class folder name
class_to_pairs = defaultdict(list)
for img_path in all_images:
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    img_dir = os.path.dirname(img_path)
    label_path = os.path.join(img_dir, f"{base_name}.txt")

    # The logo brand name is the name of the immediate parent folder
    class_name = os.path.basename(img_dir)

    if os.path.exists(label_path):
        class_to_pairs[class_name].append((img_path, label_path))

# Execute splitting and copy files
segmented_count = 0
for class_name, pairs in class_to_pairs.items():
    random.seed(42)
    random.shuffle(pairs)

    n_samples = len(pairs)
    if n_samples == 0:
        continue

    # Split indexes (80% train, 10% val, 10% test)
    val_idx = int(n_samples * 0.8)
    test_idx = int(n_samples * 0.9)

    train_pairs = pairs[:val_idx]
    val_pairs = pairs[val_idx:test_idx]
    test_pairs = pairs[test_idx:]

    # Fallback to keep val/test splits filled for tiny classes
    if len(val_pairs) == 0 and n_samples > 1:
        val_pairs = [pairs[-1]]
        train_pairs = pairs[:-1]

    def copy_files(pairs_list, split_name):
        for img, lbl in pairs_list:
            shutil.copy2(img, os.path.join(DATASET_ROOT, 'images', split_name, os.path.basename(img)))
            shutil.copy2(lbl, os.path.join(DATASET_ROOT, 'labels', split_name, os.path.basename(lbl)))

    copy_files(train_pairs, 'train')
    copy_files(val_pairs, 'val')
    copy_files(test_pairs, 'test')

    print(f"Class '{class_name}': {len(train_pairs)} train, {len(val_pairs)} val, {len(test_pairs)} test.")
    segmented_count += 1

print(f"Successfully segmented {segmented_count} classes")

Class 'trinium': 4 train, 1 val, 1 test.
Class 'thomson reuters': 60 train, 7 val, 8 test.
Class 'orkan bensin': 12 train, 2 val, 2 test.
Class 'abus': 101 train, 13 val, 13 test.
Class 'Aethra': 38 train, 5 val, 5 test.
Class 'happigo': 17 train, 2 val, 3 test.
Class 'CMD': 6 train, 1 val, 1 test.
Class 'skeena': 8 train, 1 val, 1 test.
Class 'Copeland's': 8 train, 1 val, 1 test.
Class 'southworth': 8 train, 1 val, 2 test.
Class 'vidyo-2': 36 train, 5 val, 5 test.
Class 'Gurkha-2': 15 train, 2 val, 2 test.
Class 'socar-2': 14 train, 2 val, 2 test.
Class 'spira': 8 train, 1 val, 2 test.
Class 'sf express': 50 train, 6 val, 7 test.
Class 'pulser': 80 train, 10 val, 10 test.
Class 'knudsen': 55 train, 7 val, 7 test.
Class 'tubao-1': 36 train, 5 val, 5 test.
Class 'Isosource': 19 train, 2 val, 3 test.
Class 'Cosan': 4 train, 1 val, 1 test.
Class 'sneaky pete's': 6 train, 1 val, 1 test.
Class 'socar-1': 14 train, 2 val, 2 test.
Class 'profex': 8 train, 1 val, 2 test.
Class 'lefeng': 30 tra

### Step 6: Generate Dataset Configuration YAML

In [6]:
# Generate unique sorted classes mapped to integer indices
unique_classes = sorted(list(class_to_pairs.keys()))
if not unique_classes:
    unique_classes = ["logo"]

class_dict = {i: name for i, name in enumerate(unique_classes)}

yaml_data = {
    'path': DATASET_ROOT,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': class_dict
}

yaml_path = os.path.join(DATASET_ROOT, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print(f"YOLO dataset configuration file saved to: {yaml_path}")

YOLO dataset configuration file saved to: /content/logo_yolo_dataset/dataset.yaml


### Step 7: Train the YOLO Detector

In [8]:
print("Initializing pre-trained YOLOv8 nano model...")
model = YOLO('yolov8n.pt')

# Train the model with checkpoint saving enabled
# By default, YOLOv8 saves checkpoints in {project}/{name}/weights/last.pt and best.pt
results = model.train(
    data=yaml_path,
    epochs=20,          # Adjust depending on your performance preferences and dataset complexity
    imgsz=640,
    batch=16,
    device=0,          # Set to 0 if utilizing a GPU runtime
    save=True,         # This guarantees that checkpoint weights (.pt files) are saved
    project='logo_detection',
    name='yolov8_logos'
)

print("Training process finished!")

# --- HOW TO RESUME FROM A CHECKPOINT ---
# If your training is interrupted, you can resume it by uncommenting and running the code below:
# model = YOLO('/content/logo_detection/yolov8_logos/weights/last.pt')
# model.train(resume=True)

Initializing pre-trained YOLOv8 nano model...
Ultralytics 8.4.165 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/logo_yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yol

### Step 8: Inference on Custom Test Images
This section sets up a local test directory where you can upload your own custom images. It runs the trained YOLO model on those images, uses OpenCV (`cv2`) to draw bounding boxes and class labels directly on the images, and outputs both the visual image results and the corresponding bounding box coordinate `.txt` files.

In [9]:
import cv2
from google.colab import files

# --- CONFIGURATION ---
TEST_UPLOAD_DIR = "/content/custom_test_images"
INFERENCE_OUTPUT_DIR = "/content/custom_test_results"

# Prepare folders
os.makedirs(TEST_UPLOAD_DIR, exist_ok=True)
os.makedirs(INFERENCE_OUTPUT_DIR, exist_ok=True)

print(f"Test upload directory is ready at: {TEST_UPLOAD_DIR}")
print(f"Inference results will be saved to: {INFERENCE_OUTPUT_DIR}")

Test upload directory is ready at: /content/custom_test_images
Inference results will be saved to: /content/custom_test_results


#### Run this cell to upload your custom logo images to the test folder:

In [ ]:
uploaded = files.upload()
for filename in uploaded.keys():
    shutil.move(filename, os.path.join(TEST_UPLOAD_DIR, filename))
print(f"Uploaded {len(uploaded)} images to {TEST_UPLOAD_DIR}")

#### Run the detector, draw bounding boxes, and save outputs:

In [12]:
# 1. Load the best trained model (using trained weights if available, fallback to basic nano)
model_path = '/content/logo_detection/yolov8_logos/weights/best.pt'
if os.path.exists(model_path):
    test_model = YOLO(model_path)
    print(f"Loaded custom trained model weights from: {model_path}")
else:
    test_model = model
    print("Using the currently active trained model instance.")

# 2. Get all images in test folder
test_images = []
for ext in ('*.png', '*.jpg', '*.jpeg', '*.JPG', '*.JPEG', '*.PNG'):
    test_images.extend(glob.glob(os.path.join(TEST_UPLOAD_DIR, ext)))

if len(test_images) == 0:
    print(f"No images found in {TEST_UPLOAD_DIR}. Please upload some files in the step above.")
else:
    # 3. Perform prediction and annotate
    for img_path in test_images:
        base_name = os.path.splitext(os.path.basename(img_path))[0]

        # Load image with OpenCV
        img = cv2.imread(img_path)
        h, w, _ = img.shape

        # Run YOLO inference with a lowered confidence threshold of 0.10 to help catch detections
        results = test_model.predict(source=img_path, conf=0.10, verbose=False)

        txt_output_path = os.path.join(INFERENCE_OUTPUT_DIR, f"{base_name}_predictions.txt")
        detections_found = []

        with open(txt_output_path, 'w') as txt_file:
            for r in results:
                boxes = r.boxes
                for box in boxes:
                    # Get coordinate values
                    coords = box.xyxy[0].tolist() # Left, Top, Right, Bottom
                    cls_id = int(box.cls[0].item())
                    conf = float(box.conf[0].item())
                    cls_name = test_model.names.get(cls_id, f"Class_{cls_id}")

                    detections_found.append((cls_name, conf))
                    x1, y1, x2, y2 = map(int, coords)

                    # Save coordinates in normalized format to the .txt file: [class_id center_x center_y width height]
                    dw = 1.0 / w
                    dh = 1.0 / h
                    cx = (x1 + x2) / 2.0 * dw
                    cy = (y1 + y2) / 2.0 * dh
                    bw = (x2 - x1) * dw
                    bh = (y2 - y1) * dh
                    txt_file.write(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")

                    # Draw bounding box on the image using OpenCV
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)

                    # Add class and confidence label
                    label_str = f"{cls_name}: {conf:.2f}"
                    cv2.putText(img, label_str, (x1, max(y1 - 10, 15)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        # Save the annotated image
        annotated_img_path = os.path.join(INFERENCE_OUTPUT_DIR, f"{base_name}_result.jpg")
        cv2.imwrite(annotated_img_path, img)

        if len(detections_found) > 0:
            print(f"\nProcessed {os.path.basename(img_path)} -> Detections saved to {INFERENCE_OUTPUT_DIR}")
            print("Detections:")
            for cls_name, conf in detections_found:
                print(f" - {cls_name}: {conf:.4f} confidence")
        else:
            print(f"Processed {os.path.basename(img_path)} -> No logos detected above confidence threshold 0.10. {txt_output_path} is empty.")

Using the currently active trained model instance.
Processed TeamViewer-logo.png -> No logos detected above confidence threshold 0.10. /content/custom_test_results/TeamViewer-logo_predictions.txt is empty.
Processed man-utd-authentic-home-shirt-22-23.jpg -> No logos detected above confidence threshold 0.10. /content/custom_test_results/man-utd-authentic-home-shirt-22-23_predictions.txt is empty.
